# LAB 24 — Список як Архітектурне Рішення: Taxi Dispatch System

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Що ми будуємо?

Ми симулюємо **систему диспетчеризації таксі** на реальних даних NYC Yellow Taxi.
Кожна поїздка — один елемент у черзі. Система обробляє поїздки в порядку FIFO.

**Центральне питання цієї лабораторії:**
> Яку структуру даних обрати для черги — `list`, `LinkedList` чи `deque`?
> І чому це має значення в production?

---

## Карта лабораторії

| Частина | Тема | Складність |
|---------|------|------------|
| 1 | Завантаження даних через DuckDB | Архітектура |
| 2 | Наївна черга на `list.pop(0)` | ⚠️ O(n) |
| 3 | Черга на `LinkedList` | O(1) теоретично |
| 4 | Черга на `collections.deque` | ✅ O(1) в C |
| 5 | Performance benchmark | Вимірювання |
| 6 | Аналітика поїздок | Реалізм |
| 7 | Фінальний інсайт | Архітектура |

---

> **Принцип:** `data size ≠ data loaded into memory`.
> Ми працюємо з файлом із мільйонами рядків, але тримаємо в RAM лише вибірку.

---

## Частина 1: Завантаження Даних — DuckDB Lazy Architecture

---

### Архітектура шарів

```
NYC Parquet (3M+ рядків, хмара)
        ↓  HTTPS + httpfs extension
DuckDB VIEW «trips»   ← визначає ЯК читати, але не читає нічого
        ↓  SQL SAMPLE {n} ROWS
pandas DataFrame      ← лише n рядків у RAM (5 000–10 000)
        ↓  .to_dict('records')
Python list[dict]     ← наші «поїздки» для симуляції черги
```

**Чому не `pd.read_parquet(url)`?**  
- `pandas` завантажить **весь файл** (3M рядків × 20 колонок) → ~1.5 GB RAM  
- DuckDB `SAMPLE` — reservoir sampling на рівні parquet pages → ~0.6 MB RAM  

```
DuckDB VIEW — це «рецепт», не «страва».
Страва готується лише коли ви виконуєте SELECT.
```

In [ ]:
# ── Залежності ─────────────────────────────────────────────────────────────
# pip install duckdb pandas matplotlib ipywidgets  (якщо потрібно)

import time
import timeit
from collections import deque

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── DuckDB: підключення в памʼяті ──────────────────────────────────────────
con = duckdb.connect(database=":memory:")

# httpfs — дозволяє читати parquet прямо по HTTPS
try:
    con.execute("INSTALL httpfs; LOAD httpfs;")
    print("✓ httpfs завантажено")
except Exception as e:
    print(f"  httpfs вже присутній: {e}")

# ── Lazy VIEW — жодних даних ще не завантажено ──────────────────────────────
DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2023-01.parquet"
)

con.execute(f"""
    CREATE OR REPLACE VIEW trips AS
    SELECT
        CAST(payment_type  AS INTEGER) AS payment_type,
        CAST(trip_distance AS DOUBLE)  AS trip_distance,
        CAST(PULocationID  AS INTEGER) AS PULocationID,
        CAST(DOLocationID  AS INTEGER) AS DOLocationID
    FROM read_parquet('{DATA_URL}')
    WHERE
        payment_type IN (1, 2)
        AND trip_distance > 0
        AND trip_distance < 500
""")

print("✓ VIEW 'trips' зареєстровано (lazy — жодних даних ще в RAM)")

In [ ]:
# ── SQL SAMPLE: завантажуємо лише 5 000 рядків ─────────────────────────────
# USING SAMPLE n ROWS — reservoir sampling всередині DuckDB.
# pandas НІКОЛИ не бачить повний датасет — лише результат.

SAMPLE_SIZE = 5_000

print(f"Виконуємо SQL SAMPLE {SAMPLE_SIZE:,} ROWS...")
t0 = time.perf_counter()

df_sample = con.execute(f"""
    SELECT
        CASE payment_type WHEN 1 THEN 'Card' ELSE 'Cash' END AS payment_type,
        trip_distance,
        PULocationID,
        DOLocationID
    FROM trips
    USING SAMPLE {SAMPLE_SIZE} ROWS
""").df().reset_index(drop=True)

t1 = time.perf_counter()

# Конвертуємо в list[dict] — «стрічка поїздок» для диспетчера
trips_raw = df_sample.to_dict('records')

print(f"✓ Завантажено за {(t1-t0)*1000:.0f} мс")
print(f"  Рядків у пам'яті: {len(trips_raw):,}  (з 3M+ у хмарі)")
print(f"  RAM estimate: ~{len(trips_raw) * 120 / 1024:.1f} KB")
print()
print("Перші 3 поїздки:")
for i, t in enumerate(trips_raw[:3]):
    print(f"  [{i}] {t}")

In [ ]:
# ── Статистика датасету — SQL aggregation, без повного завантаження ────────
stats_row = con.execute("""
    SELECT
        COUNT(*)                                           AS total_rows,
        SUM(CASE WHEN payment_type = 1 THEN 1 ELSE 0 END) AS card_count,
        SUM(CASE WHEN payment_type = 2 THEN 1 ELSE 0 END) AS cash_count,
        ROUND(AVG(trip_distance), 2)                       AS avg_distance_mi,
        COUNT(DISTINCT PULocationID)                       AS unique_pu_zones
    FROM trips
""").fetchone()

print("=== Статистика ПОВНОГО датасету (обчислено в DuckDB, не в Python) ===")
print(f"  Всього поїздок у хмарі : {stats_row[0]:>10,}")
print(f"  Платіж карткою         : {stats_row[1]:>10,}")
print(f"  Платіж готівкою        : {stats_row[2]:>10,}")
print(f"  Середня відстань (mi)  : {stats_row[3]:>10.2f}")
print(f"  Унікальних зон виїзду  : {stats_row[4]:>10,}")
print()
print(f"  ↑ Все це без завантаження DataFrame у RAM!")
print(f"  У нашій вибірці: {len(trips_raw):,} рядків")

---

## Частина 2: Наївна Реалізація — `list.pop(0)`

---

### Сценарій: Диспетчер приймає поїздки

Уявіть чергу таксі: машини прибувають, встають у хвіст черги.
Диспетчер обробляє з голови (FIFO).

Перша ідея джуніора:

```python
queue = list(trips)
while queue:
    trip = queue.pop(0)   # забираємо першу поїздку
    process(trip)
```

### ⚠️ Прихована катастрофа: `pop(0)` — це O(n)

```
До pop(0):  [A][B][C][D][E]  (масив вказівників в RAM)

Видаляємо A:
  CPU мусить ФІЗИЧНО ЗСУНУТИ всі вказівники вліво:
  [B][C][D][E][ ]  ← 4 операції зсуву

При 10,000 елементах → 10,000 операцій зсуву КОЖНОГО разу.
При 100,000 елементах → 100,000 операцій.
Загальна складність обробки: O(n²)
```

Це не теорія. Ви **фізично** чекаєте поки RAM перезаписується.

In [ ]:
# ── Наївна реалізація: ListDispatcher ──────────────────────────────────────

class ListDispatcher:
    """
    FIFO-черга на основі Python list.
    enqueue: list.append()  → O(1) амортизовано
    dequeue: list.pop(0)    → O(n) ← BOTTLENECK
    """

    def __init__(self, trips: list):
        self._queue: list = list(trips)   # копія — не мутуємо оригінал
        self.processed = 0

    def dispatch_all(self) -> float:
        """Обробляє всю чергу. Повертає час у секундах."""
        t0 = time.perf_counter()
        while self._queue:
            trip = self._queue.pop(0)     # ⚠️ O(n) — зсув усього масиву
            self.processed += 1
        return time.perf_counter() - t0


# ── Запуск ─────────────────────────────────────────────────────────────────
dispatcher_list = ListDispatcher(trips_raw)
elapsed_list = dispatcher_list.dispatch_all()

print("=== ListDispatcher (list.pop(0)) ===")
print(f"  Поїздок оброблено : {dispatcher_list.processed:,}")
print(f"  Час виконання     : {elapsed_list*1000:.1f} мс")
print(f"  Середньо на trip  : {elapsed_list/dispatcher_list.processed*1e6:.2f} мкс")
print()
print("  Чому повільно? Кожен pop(0) зсуває весь масив вказівників.")
print("  При 5000 елементів → ~5000 × 5000 / 2 = 12.5M операцій зсуву.")

---

## Частина 3: Черга на Зв'язному Списку — O(1) Видалення

---

### Ідея: не зсувати — перемикати вказівники

```
Черга: head → [TripA] → [TripB] → [TripC] ← tail

dequeue():
  data = head.data          # зчитуємо дані
  head = head.next          # зміщуємо вказівник — один рядок!
  # TripA стає «островом» → GC прибере

Результат: head → [TripB] → [TripC] ← tail
```

**Жоден елемент фізично не переміщується.** `O(1)` — завжди, незалежно від розміру черги.

### Архітектура: два вказівники

```
head → [A] → [B] → [C] → None
                          ↑
                         tail

enqueue(D): tail.next = D; tail = D   → O(1)
dequeue():  data = head.data; head = head.next  → O(1)
```

Потрібно відстежувати `head` і `tail` — для O(1) з обох боків.

In [ ]:
# ── Linked List Queue ───────────────────────────────────────────────────────

class _Node:
    """Приватний вузол черги. Клієнт не знає про його існування."""
    __slots__ = ('data', 'next')

    def __init__(self, data):
        self.data = data
        self.next = None


class LinkedQueue:
    """
    FIFO-черга на зв'язному списку.
    enqueue: O(1) — чіпляємо до tail
    dequeue: O(1) — переміщуємо head
    """

    def __init__(self):
        self._head: _Node | None = None
        self._tail: _Node | None = None
        self._size: int = 0

    def enqueue(self, data) -> None:
        node = _Node(data)
        if self._tail is None:          # порожня черга
            self._head = node
            self._tail = node
        else:
            self._tail.next = node      # старий tail → новий вузол
            self._tail = node           # tail = новий вузол
        self._size += 1

    def dequeue(self):
        if self._head is None:
            raise IndexError("dequeue from empty queue")
        data = self._head.data
        self._head = self._head.next    # переміщуємо вказівник — O(1)
        if self._head is None:
            self._tail = None
        self._size -= 1
        return data

    def is_empty(self) -> bool: return self._head is None
    def __len__(self) -> int:   return self._size


# ── LinkedDispatcher ────────────────────────────────────────────────────────

class LinkedDispatcher:
    """
    FIFO-черга на LinkedQueue.
    Всі операції enqueue/dequeue — O(1).
    """

    def __init__(self, trips: list):
        self._queue = LinkedQueue()
        for trip in trips:
            self._queue.enqueue(trip)
        self.processed = 0

    def dispatch_all(self) -> float:
        t0 = time.perf_counter()
        while not self._queue.is_empty():
            trip = self._queue.dequeue()  # O(1) — лише зсув вказівника
            self.processed += 1
        return time.perf_counter() - t0


# ── Запуск ─────────────────────────────────────────────────────────────────
dispatcher_linked = LinkedDispatcher(trips_raw)
elapsed_linked = dispatcher_linked.dispatch_all()

print("=== LinkedDispatcher (LinkedQueue.dequeue) ===")
print(f"  Поїздок оброблено : {dispatcher_linked.processed:,}")
print(f"  Час виконання     : {elapsed_linked*1000:.1f} мс")
print(f"  Середньо на trip  : {elapsed_linked/dispatcher_linked.processed*1e6:.2f} мкс")
print()
if elapsed_list > 0:
    print(f"  Порівняно з list.pop(0): {elapsed_list/elapsed_linked:.1f}x швидше")

---

## Частина 4: `collections.deque` — Production Solution

---

### Чому не просто використати наш LinkedQueue?

Наш `LinkedQueue` — чудовий навчальний приклад, але в production є нюанси:

| Критерій | `LinkedQueue` (Python) | `collections.deque` (C) |
|----------|------------------------|-------------------------|
| Мова реалізації | CPython bytecode | Скомпільований C |
| Overhead на вузол | `_Node` object (~56B) + GC | Блок фіксованих 64-node chunks |
| `appendleft/popleft` | O(1) | O(1) |
| `append/pop` | O(n)/O(1) | O(1) |
| Thread-safe операції | Ні | GIL-safe для append/pop |
| `maxlen` (кільцевий буфер) | Немає | Вбудований |

### Архітектура `deque`

`deque` — **не** чистий зв'язний список і **не** масив.
Це **doubly linked list of fixed-size blocks** (64 вузли на блок).

```
Block 0: [e0][e1]...[e63]  ←→  Block 1: [e64]...[e127]  ←→  Block 2: ...
```

Блоки — суцільні в RAM (cache-friendly всередині блоку).
Перехід між блоками — через вказівник (linked).
Краще за чистий linked list для ітерації, краще за масив для вставок на кінцях.

In [ ]:
# ── DequeDispatcher ─────────────────────────────────────────────────────────

class DequeDispatcher:
    """
    FIFO-черга на collections.deque.
    append:    O(1) — правий кінець
    popleft:   O(1) — лівий кінець  ← замість pop(0)
    """

    def __init__(self, trips: list):
        self._queue: deque = deque(trips)   # O(n) одноразово при ініціалізації
        self.processed = 0

    def dispatch_all(self) -> float:
        t0 = time.perf_counter()
        while self._queue:
            trip = self._queue.popleft()    # O(1) в C — набагато швидше
            self.processed += 1
        return time.perf_counter() - t0


# ── Запуск ─────────────────────────────────────────────────────────────────
dispatcher_deque = DequeDispatcher(trips_raw)
elapsed_deque = dispatcher_deque.dispatch_all()

print("=== DequeDispatcher (deque.popleft) ===")
print(f"  Поїздок оброблено : {dispatcher_deque.processed:,}")
print(f"  Час виконання     : {elapsed_deque*1000:.1f} мс")
print(f"  Середньо на trip  : {elapsed_deque/dispatcher_deque.processed*1e6:.2f} мкс")
print()
print("─" * 55)
print(f"  list.pop(0)   : {elapsed_list*1000:8.1f} мс")
print(f"  LinkedQueue   : {elapsed_linked*1000:8.1f} мс")
print(f"  deque.popleft : {elapsed_deque*1000:8.1f} мс")
print("─" * 55)

---

## Частина 5: Performance Benchmark — Де Проявляється Різниця

---

### ⚠️ Чому 5000 рядків — недостатньо для демонстрації?

При малих наборах `list.pop(0)` здається прийнятним.  
Але `O(n²)` — це **квадратична** залежність: подвоїли розмір → **учетверо** більше роботи.

Ось що ми зараз поміряємо:

```
n = 1,000  → list: ~1M зсувів
n = 3,000  → list: ~9M зсувів  (×9 при ×3 даних)
n = 5,000  → list: ~25M зсувів
n = 10,000 → list: ~100M зсувів
```

Тоді як `deque` і `LinkedQueue`: **лінійно** — тільки `n` операцій.

In [ ]:
# ── Benchmark helper ────────────────────────────────────────────────────────

def bench_list(n: int, trips_pool: list) -> float:
    """Час обробки n поїздок через list.pop(0)."""
    sample = (trips_pool * (n // len(trips_pool) + 1))[:n]
    q = list(sample)
    t0 = time.perf_counter()
    while q:
        q.pop(0)
    return time.perf_counter() - t0


def bench_linked(n: int, trips_pool: list) -> float:
    """Час обробки n поїздок через LinkedQueue.dequeue."""
    sample = (trips_pool * (n // len(trips_pool) + 1))[:n]
    q = LinkedQueue()
    for trip in sample:
        q.enqueue(trip)
    t0 = time.perf_counter()
    while not q.is_empty():
        q.dequeue()
    return time.perf_counter() - t0


def bench_deque(n: int, trips_pool: list) -> float:
    """Час обробки n поїздок через deque.popleft."""
    sample = (trips_pool * (n // len(trips_pool) + 1))[:n]
    q = deque(sample)
    t0 = time.perf_counter()
    while q:
        q.popleft()
    return time.perf_counter() - t0


# ── Запускаємо benchmark ────────────────────────────────────────────────────
SIZES = [1_000, 3_000, 5_000, 10_000]
REPEATS = 3   # середнє по кількох запусках

results = {"n": [], "list": [], "linked": [], "deque": []}

print(f"{'n':>8} | {'list (мс)':>12} | {'linked (мс)':>13} | {'deque (мс)':>12}")
print("-" * 55)

for n in SIZES:
    t_list   = min(bench_list(n, trips_raw)   for _ in range(REPEATS))
    t_linked = min(bench_linked(n, trips_raw) for _ in range(REPEATS))
    t_deque  = min(bench_deque(n, trips_raw)  for _ in range(REPEATS))

    results["n"].append(n)
    results["list"].append(t_list   * 1000)
    results["linked"].append(t_linked * 1000)
    results["deque"].append(t_deque  * 1000)

    print(f"{n:>8,} | {t_list*1000:>12.2f} | {t_linked*1000:>13.2f} | {t_deque*1000:>12.2f}")

print()
print("Одиниці: мілісекунди (менше = краще)")

In [ ]:
# ── Візуалізація ─────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Taxi Dispatch System: Performance Comparison", fontsize=14, fontweight='bold')

colors  = {"list": "#e74c3c", "linked": "#f39c12", "deque": "#27ae60"}
markers = {"list": "o",       "linked": "s",        "deque": "^"}
labels  = {
    "list":   "list.pop(0)  --  O(n)",
    "linked": "LinkedQueue  --  O(1) Python",
    "deque":  "deque.popleft  --  O(1) C",
}

# ── Графік 1: всі три структури ──────────────────────────────────────────
ax1 = axes[0]
for key in ["list", "linked", "deque"]:
    ax1.plot(results["n"], results[key],
             color=colors[key],
             marker=markers[key],
             linewidth=2,
             markersize=7,
             label=labels[key])

ax1.set_xlabel("Кількість поїздок у черзі", fontsize=11)
ax1.set_ylabel("Час виконання (мс)", fontsize=11)
ax1.set_title("Повне порівняння (включно з list)", fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Анотація — без emoji (Arial на Windows не має гліфів Unicode emoji)
ax1.annotate(
    "[!] O(n^2) bottleneck",
    xy=(results["n"][-1], results["list"][-1]),
    xytext=(results["n"][-2], results["list"][-1] * 0.7),
    fontsize=9, color=colors["list"],
    arrowprops=dict(arrowstyle="->", color=colors["list"], lw=1.5)
)

# ── Графік 2: тільки linked vs deque (зум без list) ──────────────────────
ax2 = axes[1]
for key in ["linked", "deque"]:
    ax2.plot(results["n"], results[key],
             color=colors[key],
             marker=markers[key],
             linewidth=2,
             markersize=7,
             label=labels[key])

ax2.set_xlabel("Кількість поїздок у черзі", fontsize=11)
ax2.set_ylabel("Час виконання (мс)", fontsize=11)
ax2.set_title("Zoom: LinkedQueue vs deque (без list)", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

ax2.text(0.05, 0.95,
         "Обидва O(1), але deque\nреалізований у C --\n~10-50x менший overhead",
         transform=ax2.transAxes,
         fontsize=8, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()


In [ ]:
# ── Таблиця: прискорення deque відносно list ────────────────────────────────

print("=== Speedup deque.popleft відносно list.pop(0) ===")
print()
print(f"{'n':>8} | {'list (мс)':>10} | {'deque (мс)':>11} | {'speedup':>8} | {'Складність'}")
print("-" * 62)

for i, n in enumerate(results["n"]):
    speedup = results["list"][i] / results["deque"][i] if results["deque"][i] > 0 else float('inf')
    print(
        f"{n:>8,} | "
        f"{results['list'][i]:>10.2f} | "
        f"{results['deque'][i]:>11.3f} | "
        f"{speedup:>7.1f}x | "
        f"{'O(n²) vs O(n)'}"
    )

print()
print("Ключовий інсайт: при ×10 даних → list сповільнюється в ~×100,")
print("тоді як deque — лінійно в ~×10. Це і є різниця O(n²) vs O(n).")

---

## Частина 6: Системна Інтерпретація — Чому Це Важливо

---

### `list.pop(0)` — чому провалюється в production

```
Celery task queue: 100,000 tasks

З list.pop(0):
  обробка 1-ї задачі:   зсуває 99,999 вказівників
  обробка 2-ї задачі:   зсуває 99,998 вказівників
  ...
  Всього: ~5 × 10⁹ операцій → хвилини замість секунд

З deque.popleft:
  кожна операція: 1 вказівник → завжди ~1 мкс
```

---

### `LinkedList` — теоретично добре, практично складно

Python-реалізація `_Node` має **overhead**:
- Кожен `_Node` — Python object (~56 байт + GC overhead)
- При 100k елементів: ~5.6 MB тільки на вузлові об'єкти
- Python GC сканує всі ці об'єкти → додаткові паузи

Плюс: розкидані вузли → **cache miss** на кожному переході.

---

### `collections.deque` — production ready

Чому deque перемагає навіть наш O(1) LinkedQueue:

| Фактор | LinkedQueue (Python) | deque (C) |
|--------|---------------------|----------|
| Виклик методу | Python bytecode dispatch | прямий C виклик |
| Виділення пам'яті | `_Node.__new__` кожного разу | блоки по 64 елементи |
| GC pressure | Окремий tracked object | Мінімальний |
| Cache locality | Кожен вузол — своя адреса | 64 елементи поруч |

---

### Правило вибору структури для черги

```
Потрібна черга (FIFO)?
├─ ТАК → collections.deque  (завжди, в production)
│        або asyncio.Queue  (в async коді)
│        або queue.Queue    (multi-threading)
│
└─ Навчальні цілі → LinkedQueue  (розуміння механіки)

НЕ РОБІТЬ: list.pop(0) для черги
```

---

## Частина 7: Аналітика Поїздок — Реалістичний Контекст

---

Тепер що ми обробили всі поїздки через чергу — проаналізуємо дані.
Це показує, що структури даних — не самоціль, а **інструмент** для реальних задач.

In [ ]:
# ── Аналітика на вибірці df_sample ─────────────────────────────────────────

# Загальна та середня відстань
total_distance  = df_sample["trip_distance"].sum()
avg_distance    = df_sample["trip_distance"].mean()
median_distance = df_sample["trip_distance"].median()
max_distance    = df_sample["trip_distance"].max()

# Cash vs Card
payment_counts = df_sample["payment_type"].value_counts()
cash_pct  = payment_counts.get("Cash",  0) / len(df_sample) * 100
card_pct  = payment_counts.get("Card", 0) / len(df_sample) * 100

# Топ-5 зон посадки
top_pickup = df_sample["PULocationID"].value_counts().head(5)

print(f"=== Аналітика {len(df_sample):,} поїздок (NYC Yellow Taxi, Jan 2023) ===")
print()
print("📍 Відстані (милі):")
print(f"   Загальна : {total_distance:>10,.1f} миль")
print(f"   Середня  : {avg_distance:>10.2f} миль")
print(f"   Медіана  : {median_distance:>10.2f} миль")
print(f"   Максимум : {max_distance:>10.1f} миль")
print()
print("💳 Спосіб оплати:")
for method, count in payment_counts.items():
    pct = count / len(df_sample) * 100
    bar = "█" * int(pct / 2)
    print(f"   {method:>6}: {count:>6,} поїздок ({pct:.1f}%)  {bar}")
print()
print("📍 Топ-5 зон посадки (LocationID):")
for zone_id, count in top_pickup.items():
    print(f"   Зона {zone_id:>4}: {count:>4} поїздок")

In [ ]:
# ── Гістограма розподілу відстаней ──────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("NYC Taxi Trips Analytics", fontsize=13, fontweight='bold')

# Гістограма відстаней (обрізаємо до 95-го перцентилю для читабельності)
p95 = df_sample["trip_distance"].quantile(0.95)
dist_filtered = df_sample[df_sample["trip_distance"] <= p95]["trip_distance"]

axes[0].hist(dist_filtered, bins=40, color="#3498db", edgecolor="white", alpha=0.85)
axes[0].axvline(avg_distance, color="#e74c3c", linestyle="--", linewidth=1.5,
                label=f"Середня: {avg_distance:.2f} mi")
axes[0].axvline(median_distance, color="#27ae60", linestyle="-.", linewidth=1.5,
                label=f"Медіана: {median_distance:.2f} mi")
axes[0].set_xlabel("Відстань (милі)", fontsize=10)
axes[0].set_ylabel("Кількість поїздок", fontsize=10)
axes[0].set_title(f"Розподіл відстаней (≤{p95:.1f} mi, 95-й перц.)", fontsize=10)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Pie chart: Cash vs Card
axes[1].pie(
    payment_counts.values,
    labels=[f"{m}\n{c:,} ({c/len(df_sample)*100:.1f}%)" for m, c in payment_counts.items()],
    colors=["#27ae60", "#3498db"],
    autopct=None,
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[1].set_title("Спосіб оплати", fontsize=10)

plt.tight_layout()
plt.show()

---

## Частина 8: Фінальний Інсайт — Структура Даних = Рішення

---

### Ланцюжок причинності

```
Структура даних
      ↓
Модель пам'яті  (суцільний блок? вузли-острівці? гібрид?)
      ↓
Складність операцій  (O(1)? O(n)? O(n²)?)
      ↓
Час виконання  (мілісекунди? секунди? хвилини?)
      ↓
Масштабованість системи  (100 RPS? 10,000 RPS? відмова під навантаженням?)
```

---

### Коли що обирати

| Задача | Структура | Чому |
|--------|-----------|------|
| Черга (FIFO) | `deque` | O(1) з обох кінців, C-реалізація |
| Стек (LIFO) | `list` | `append/pop` — обидва O(1) |
| Довільний доступ `[i]` | `list` | O(1) — масив вказівників |
| Часті вставки на початку | `deque` | O(1) vs O(n) у list |
| Обмежена черга | `deque(maxlen=N)` | Кільцевий буфер без ручного видалення |
| Паралельна черга | `queue.Queue` | Thread-safe |
| Async черга | `asyncio.Queue` | Coroutine-safe |

---

### Зв'язний список: де він реально корисний?

```python
# LRU Cache (dict + deque): O(1) get і put
from functools import lru_cache

# collections.OrderedDict — під капотом doubly linked list
# asyncio event loop — queue реалізована на deque
# CPython frame stack — call stack — linked list
```

Зв'язний список не зникає — він **захований** всередині стандартної бібліотеки.

---

### Головний висновок

> **`list.pop(0)` у черзі — це не "трохи повільно".**  
> Це архітектурна помилка, яка масштабується квадратично.  
> При 10× більше даних — система сповільнюється в ~100×.
>
> Розуміння фізичної моделі пам'яті — це те, що відрізняє  
> senior-інженера від junior: **вибір структури до написання коду**.

---

## Бонус: Інтерактивний Слайдер

---

Запустіть клітинку нижче і перетягніть слайдер, щоб побачити як час змінюється  
при різних розмірах вибірки в реальному часі.

> **Потрібно:** `pip install ipywidgets` і Jupyter з увімкненим widgets extension.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    output = widgets.Output()

    slider = widgets.IntSlider(
        value=2_000,
        min=500,
        max=12_000,
        step=500,
        description='Розмір N:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )

    label = widgets.Label(value="")

    def on_slider_change(change):
        n = change['new']
        t_l = bench_list(n, trips_raw)
        t_k = bench_linked(n, trips_raw)
        t_d = bench_deque(n, trips_raw)
        speedup = t_l / t_d if t_d > 0 else 0
        label.value = (
            f"  list: {t_l*1000:.2f} мс  │  "
            f"linked: {t_k*1000:.2f} мс  │  "
            f"deque: {t_d*1000:.3f} мс  │  "
            f"speedup deque/list: {speedup:.1f}×"
        )

    slider.observe(on_slider_change, names='value')
    on_slider_change({'new': slider.value})   # initial run

    display(widgets.VBox([
        widgets.HTML("<b>Перетягніть слайдер — порівняйте структури в реальному часі</b>"),
        slider,
        label
    ]))

except ImportError:
    print("ipywidgets не встановлений.")
    print("Встановіть: pip install ipywidgets")
    print()
    print("Ручний запуск для кількох значень:")
    for n in [500, 2000, 5000, 10000]:
        t_l = bench_list(n, trips_raw)
        t_d = bench_deque(n, trips_raw)
        print(f"  n={n:>6,}: list={t_l*1000:7.2f} мс | deque={t_d*1000:6.3f} мс | speedup={t_l/t_d:.1f}×")

---

## Підсумок Лабораторії

---

| Що ми зробили | Чого навчилися |
|---------------|----------------|
| DuckDB lazy VIEW | Data size ≠ Data in RAM |
| `list.pop(0)` queue | O(n) = масовий зсув = O(n²) загалом |
| `LinkedQueue` | O(1) dequeue через зсув вказівника |
| `deque.popleft` | O(1) в C = production-ready |
| Benchmark 1k–10k | Квадратичний vs лінійний ріст |
| Аналітика NYC Taxi | Реальні дані = реальна інженерія |

---

> **Структура даних — це не вибір синтаксису. Це рішення про фізику пам'яті.**  
> Правильний вибір на старті → система, яка масштабується.  
> Неправильний вибір → O(n²) квадратична катастрофа під навантаженням.

---

*Урок 24 · Module 3 · Python Advanced · Viktor Nikoriak*